# Image-to-recipe training and evaluation
Use a free T4 and the Drive account containing the completed text stitch adapter. 140 authored training images plus 88 text retention examples; 28 development and 40 frozen challenge images. Continue the old adapter into a new output folder; old weights are preserved. No paid resources or automatic deployment. These are small flat references, not whole 3D objects or verified physical patterns. Run setup, then the training/evaluation cell. It saves checkpoints and predictions in Drive.

In [ ]:
import sys,subprocess,shutil,json
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No eligible free CUDA GPU. Stop; do not upgrade or bypass quotas.')
from google.colab import drive
drive.mount('/content/drive',timeout_ms=600000)
image_run=Path('/content/drive/MyDrive/Beadform-free-qwen/runs/image-stitch-v1')
image_run.mkdir(parents=True,exist_ok=True)
image_train_repo=Path('/content/beadform-image-stitch-v1')
if not image_train_repo.exists():
    subprocess.run(['git','clone','--branch','main','https://github.com/c-tustin/beadform.git',str(image_train_repo)],check=True)
if subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=image_train_repo,text=True).strip():
    raise RuntimeError('Source has edits; preserve them before proceeding.')
revision_file=image_run/'repository-revision.txt'
image_revision=revision_file.read_text().strip() if revision_file.exists() else subprocess.check_output(['git','rev-parse','HEAD'],cwd=image_train_repo,text=True).strip()
subprocess.run(['git','fetch','origin',image_revision],cwd=image_train_repo,check=True)
subprocess.run(['git','checkout','--detach',image_revision],cwd=image_train_repo,check=True)
revision_file.write_text(image_revision+chr(10))
if not shutil.which('node'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','nodejs'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(image_train_repo/'training/requirements-catalog-v2.txt')],check=True)
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
freeze_file=image_run/'pip-freeze.txt'
if freeze_file.exists() and freeze_file.read_text()!=freeze:
    raise RuntimeError('Environment changed; inspect before resuming. Saved weights remain intact.')
freeze_file.write_text(freeze)
image_parent=Path('/content/drive/MyDrive/Beadform-free-qwen/runs/stitch-program-v1/adapter')
print('Source:',image_revision,'GPU:',torch.cuda.get_device_name(0),flush=True)
check=subprocess.run([sys.executable,str(image_train_repo/'training/train_image_stitch.py'),'--output',str(image_run),'--parent-adapter',str(image_parent),'--check'],cwd=image_train_repo,capture_output=True,text=True)
print(check.stdout);print(check.stderr);check.check_returncode()


In [ ]:
command=[sys.executable,'-u',str(image_train_repo/'training/train_image_stitch.py'),'--output',str(image_run),'--parent-adapter',str(image_parent),'--resume']
if not (image_run/'comparison.json').exists():
    with (image_run/'run.log').open('a') as log:
        process=subprocess.Popen(command,cwd=image_train_repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        code=process.wait()
        if code:raise RuntimeError(f'Image training stopped with code {code}; inspect the saved log. Do not restart without inspecting checkpoints.')
print((image_run/'comparison.json').read_text())
print('Saved:',image_run)
print('No automatic deployment. Synthetic images do not establish real-photo accuracy or physical beadability.')


In [ ]:
#@title Audit completed image results and open the new adapter
import sys,subprocess,json,runpy,gzip,base64
from pathlib import Path
image_run=Path('/content/drive/MyDrive/Beadform-free-qwen/runs/image-stitch-v1')
image_train_repo=Path('/content/beadform-image-stitch-v1')
if not (image_run/'comparison.json').exists():
    raise RuntimeError('Training/evaluation has not completed. Inspect the preceding run; do not start another run.')
tools_revision_file=image_run/'result-tools-revision.txt'
requested_revision=tools_revision_file.read_text().strip() if tools_revision_file.exists() else 'main'
subprocess.run(['git','fetch','origin',requested_revision],cwd=image_train_repo,check=True,capture_output=True)
tools_revision=subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=image_train_repo,text=True).strip()
for name in ['audit_image_stitch_results.py','generate_image_stitch.py','colab_image_stitch_try.py']:
    path='training/'+name
    text=subprocess.check_output(['git','show',tools_revision+':'+path],cwd=image_train_repo,text=True)
    target=image_train_repo/path
    if target.exists() and target.read_text()!=text:
        raise RuntimeError('Preserve existing result-tool edits before replacing '+name)
    target.write_text(text)
tools_revision_file.write_text(tools_revision+chr(10))
audit_process=subprocess.run([sys.executable,str(image_train_repo/'training/audit_image_stitch_results.py'),'--run',str(image_run)],cwd=image_train_repo,capture_output=True,text=True)
if audit_process.returncode:
    print(audit_process.stdout);print(audit_process.stderr);audit_process.check_returncode()
public={name:json.loads((image_run/(name+'.json')).read_text()) for name in ['comparison','audit','metadata','training-complete']}
public['training_repository_revision']=(image_run/'repository-revision.txt').read_text().strip()
public['result_tools_revision']=tools_revision
public['predictions']={folder.name:[json.loads(line) for line in (folder/'predictions.jsonl').read_text().splitlines() if line] for folder in sorted(image_run.iterdir()) if folder.is_dir() and (folder/'predictions.jsonl').exists()}
print('BEADFORM_RESULTS_V1:'+base64.b64encode(gzip.compress(json.dumps(public).encode())).decode())
print('Audit:',public['audit']['status'],'Predictions:',public['audit']['prediction_count'])
print('Old adapter preserved. New weights saved separately. Synthetic-image evaluation only; no physical beadability claim.')
new_image_ui=runpy.run_path(str(image_train_repo/'training/colab_image_stitch_try.py'))
